In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import numpy as np
import librosa
import torch

from transformers import (
    WavLMModel,
    Wav2Vec2FeatureExtractor
)

# Step 2: Set your base directory matching your Google Drive layout
# Change "My Drive/YOUR_FOLDER_PATH" to wherever your dataset resides
BASE_DIR = "/content/drive/MyDrive/IASNLP Project/data/CosyVoice"

# ---------------------------------------------------
# Model Initialization
# ---------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    "microsoft/wavlm-large"
)

model = WavLMModel.from_pretrained(
    "microsoft/wavlm-large"
).to(device)

model.eval()

def extract_embedding(audio_path):
    # Load audio at WavLM's native 16kHz sampling rate
    speech, sr = librosa.load(audio_path, sr=16000)

    inputs = feature_extractor(
        speech,
        sampling_rate=16000,
        return_tensors="pt"
    )
    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        outputs = model(
            input_values,
            output_hidden_states=True
        )

    # shape: [num_layers (25), batch, time_steps, hidden_size]
    hidden_states = torch.stack(outputs.hidden_states)

    # Reverted back to your original setup: Average top 4 layers
    layer_averaged = hidden_states[-4:].mean(dim=0) # [batch, time_steps, 768]
    # layer_averaged = hidden_states[-1]

    # Calculate standard unweighted statistical poolings over the time axis (dim=1)
    # mean_pool = layer_averaged.mean(dim=1)
    # std_pool = layer_averaged.std(dim=1)
    # max_pool, _ = layer_averaged.max(dim=1)
    # min_pool, _ = layer_averaged.min(dim=1)

    # Concatenate all 4 structural representations
    # embedding = torch.cat([mean_pool, std_pool, max_pool, min_pool], dim=1)

    embedding = layer_averaged.mean(dim=1)

    return embedding.squeeze().cpu().numpy()


# ---------------------------------------------------
# Processing Pipeline Execution Loop
# ---------------------------------------------------
if not os.path.exists(BASE_DIR):
    print(f"Error: '{BASE_DIR}' not found.")
else:

    for emotion in os.listdir(BASE_DIR):

        emotion_path = os.path.join(
            BASE_DIR,
            emotion
        )

        if not os.path.isdir(emotion_path):
            continue

        embeddings_dir = os.path.join(
            emotion_path,
            "wavlm_large_mean_embeddings"
        )

        os.makedirs(
            embeddings_dir,
            exist_ok=True
        )

        print(f"\nProcessing Emotion: {emotion}")

        for file_name in os.listdir(emotion_path):

            file_path = os.path.join(
                emotion_path,
                file_name
            )

            # Only process WAV files
            if (
                not os.path.isfile(file_path)
                or not file_name.lower().endswith(".wav")
            ):
                continue

            save_path = os.path.join(
                embeddings_dir,
                file_name.replace(".wav", ".npy")
            )

            if os.path.exists(save_path):
                continue

            try:
                embedding = extract_embedding(
                    file_path
                )

                np.save(
                    save_path,
                    embedding
                )

                print(f"Saved -> {save_path}")

            except Exception as e:
                print(f"Error processing {file_name}")
                print(e)

    print("\nFinished Feature Extraction!")

Using device: cuda


Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]


Processing Emotion: ravdess

Processing Emotion: sad

Processing Emotion: happy

Processing Emotion: angry

Processing Emotion: surprise

Finished Feature Extraction!
